# crunchy — MCNP parsing in Python

[`crunchy`](https://github.com/AlvaroCubi/crunchy) is a fast, **lossless** MCNP
input parser written in Rust, with Python bindings. This notebook tours its
capabilities on a small self-contained deck:

1. Parse and inspect the model (cells, surfaces, materials, transforms).
2. Prove the parse is **byte-for-byte lossless**.
3. **Renumber the whole geometry** — definitions *and* every reference — and
   confirm everything else is preserved exactly.


In [1]:
import crunchy
print("crunchy version:", crunchy.__version__)


crunchy version: 0.0.0


## The input deck

A small pin-cell-ish model exercising comments, an inline `$` comment, a `&` continuation, a union (`:`), a cell complement (`#4`), a `LIKE n BUT` cell, an `RPP` macrobody, and a transformed surface.

In [2]:
DECK = 'Crunchy demo: a small pin-cell-ish model\nc --- cells ---\n1 1 -10.5 -1 imp:n=1                  $ fuel pin\n2 2 -1.0  1 -2 imp:n=1                $ water gap\n3 0 (2 -3) #4 imp:n=1                 $ moderator minus the insert\n4 3 -2.7 -40 imp:n=1 imp:p=1 &\n     vol=8.0                          $ aluminium insert (macrobody)\n5 0 3 : 20 imp:n=0                    $ graveyard\n10 like 1 but mat=4 rho=-11.3         $ second pin, different fuel\n\nc --- surfaces ---\n1 CZ 0.5\n2 CZ 0.6\n3 CZ 5.0\n20 SO 100\n30 1 PZ 50                            $ surface positioned by transform 1\n40 RPP -1 1 -1 1 -1 2\n\nc --- data ---\nm1 92235.80c 0.04 92238.80c 0.96      $ enriched uranium\nm2 1001.31c 2 8016.31c 1              $ light water\nmt2 lwtr.10t\nm3 13027.80c 1                        $ aluminium\nm4 92235.80c 0.20 92238.80c 0.80\ntr1 0 0 5\nmode n\nsdef pos=0 0 0 erg=2.0\nf4:n 1 2 4\n'
print(DECK)

Crunchy demo: a small pin-cell-ish model
c --- cells ---
1 1 -10.5 -1 imp:n=1                  $ fuel pin
2 2 -1.0  1 -2 imp:n=1                $ water gap
3 0 (2 -3) #4 imp:n=1                 $ moderator minus the insert
4 3 -2.7 -40 imp:n=1 imp:p=1 &
     vol=8.0                          $ aluminium insert (macrobody)
5 0 3 : 20 imp:n=0                    $ graveyard
10 like 1 but mat=4 rho=-11.3         $ second pin, different fuel

c --- surfaces ---
1 CZ 0.5
2 CZ 0.6
3 CZ 5.0
20 SO 100
30 1 PZ 50                            $ surface positioned by transform 1
40 RPP -1 1 -1 1 -1 2

c --- data ---
m1 92235.80c 0.04 92238.80c 0.96      $ enriched uranium
m2 1001.31c 2 8016.31c 1              $ light water
mt2 lwtr.10t
m3 13027.80c 1                        $ aluminium
m4 92235.80c 0.20 92238.80c 0.80
tr1 0 0 5
mode n
sdef pos=0 0 0 erg=2.0
f4:n 1 2 4



In [3]:
deck = crunchy.parse(DECK)          # or crunchy.Deck.from_file("model.mcnp")
deck


Deck(cells=6, surfaces=6, materials=4, transforms=1, diagnostics=0)

## Parse summary

The parser is recoverable — a clean parse yields no diagnostics.

In [4]:
print("cells:      ", deck.num_cells)
print("surfaces:   ", deck.num_surfaces)
print("materials:  ", len(deck.materials))
print("transforms: ", len(deck.transforms))
print("diagnostics:", deck.diagnostics)


cells:       6
surfaces:    6
materials:   4
transforms:  1
diagnostics: []


## Surfaces (typed)

Each surface exposes its number, mnemonic, coefficients, and an optional transform.

In [5]:
for s in deck.surfaces:
    tr = f"  (TR{s.transform})" if s.transform else ""
    print(f"{s.id:>3}  {s.kind:<4} {s.coeffs}{tr}")


  1  CZ   [0.5]
  2  CZ   [0.6]
  3  CZ   [5.0]
 20  SO   [100.0]
 30  PZ   [50.0]  (TR1)
 40  RPP  [-1.0, 1.0, -1.0, 1.0, -1.0, 2.0]


## Cells & geometry references

Cells expose material/density, void status, the surfaces they reference (with sense), cell complements, and the `LIKE n` base.

In [6]:
for c in deck.cells:
    if c.like is not None:
        print(f"{c.id:>3}  LIKE {c.like} BUT ...")
        continue
    kind = "void" if c.is_void else f"mat {c.material} @ {c.density}"
    extra = f"  complements #{c.cell_refs}" if c.cell_refs else ""
    print(f"{c.id:>3}  {kind:<14} surfaces {c.signed_surfaces}{extra}")


  1  mat 1 @ -10.5  surfaces [-1]
  2  mat 2 @ -1.0   surfaces [1, -2]
  3  void           surfaces [2, -3]  complements #[4]
  4  mat 3 @ -2.7   surfaces [-40]
  5  void           surfaces [3, 20]
 10  LIKE 1 BUT ...


## Materials & transforms

In [7]:
for m in deck.materials:
    comp = "  ".join(f"{z}={frac}" for z, frac in m.entries)
    print(f"m{m.id:<2} {comp}")

print()
for t in deck.transforms:
    print(f"tr{t.id}  displacement {t.displacement}  degrees={t.degrees}")


m1  92235.80c=0.04  92238.80c=0.96
m2  1001.31c=2.0  8016.31c=1.0
m3  13027.80c=1.0
m4  92235.80c=0.2  92238.80c=0.8

tr1  displacement (0.0, 0.0, 5.0)  degrees=False


## Fast id lookups

`deck.surface(id)` / `deck.cell(id)` use a cached index — no need to scan the whole list.

In [8]:
print(deck.surface(40))
print(deck.cell(3))
print(deck.material(1))


Surface(id=40, kind="RPP", coeffs=[-1.0, 1.0, -1.0, 1.0, -1.0, 2.0])
Cell(id=3, material=Some(0), density=None, surfaces=[2, -3])
Material(id=1, entries=2)


## Lossless round-trip

Re-emitting the deck reproduces the input **byte-for-byte** — comments, spacing, and continuations included.

In [9]:
assert str(deck) == DECK
print("byte-for-byte lossless:", str(deck) == DECK)


byte-for-byte lossless: True


## Whole-geometry renumbering

The headline capability: renumbering updates **definitions and every reference**
consistently — signed surfaces in geometry, `#n` complements, and `LIKE n`
bases. A mapping may be a callable or a `dict`.


In [10]:
def cell_line(text, cell_id):
    for line in text.splitlines():
        toks = line.split()
        if toks and toks[0] == str(cell_id):
            return line
    return None

print("cell 3 before:", cell_line(str(deck), 3))

deck.offset_surfaces(1000)            # every surface +1000 (defs + refs)
deck.renumber_cells(lambda n: n + 900)  # every cell   +900  (defs + #n + LIKE)

print("cell 3 after: ", cell_line(str(deck), 903))


cell 3 before: 3 0 (2 -3) #4 imp:n=1                 $ moderator minus the insert
cell 3 after:  903 0 (1002 -1003) #904 imp:n=1                 $ moderator minus the insert


Notice inside cell 903: surfaces `2 -3` became `1002 -1003`, the complement `#4` became `#904`, and the cell id itself `3` became `903`. Everything else on the line — the `$` comment and spacing — is untouched.

In [11]:
edited = str(deck)
print("comment preserved:      ", "$ moderator minus the insert" in edited)
print("continuation preserved: ", "imp:p=1 &" in edited)
print("surface def renumbered:  ", "1040 RPP -1 1 -1 1 -1 2" in edited)
print("LIKE base renumbered:    ", "910 like 901 but" in edited)


comment preserved:       True
continuation preserved:  True
surface def renumbered:   True
LIKE base renumbered:     True


In [12]:
print(edited)


Crunchy demo: a small pin-cell-ish model
c --- cells ---
901 1 -10.5 -1001 imp:n=1                  $ fuel pin
902 2 -1.0  1001 -1002 imp:n=1                $ water gap
903 0 (1002 -1003) #904 imp:n=1                 $ moderator minus the insert
904 3 -2.7 -1040 imp:n=1 imp:p=1 &
     vol=8.0                          $ aluminium insert (macrobody)
905 0 1003 : 1020 imp:n=0                    $ graveyard
910 like 901 but mat=4 rho=-11.3         $ second pin, different fuel

c --- surfaces ---
1001 CZ 0.5
1002 CZ 0.6
1003 CZ 5.0
1020 SO 100
1030 1 PZ 50                            $ surface positioned by transform 1
1040 RPP -1 1 -1 1 -1 2

c --- data ---
m1 92235.80c 0.04 92238.80c 0.96      $ enriched uranium
m2 1001.31c 2 8016.31c 1              $ light water
mt2 lwtr.10t
m3 13027.80c 1                        $ aluminium
m4 92235.80c 0.20 92238.80c 0.80
tr1 0 0 5
mode n
sdef pos=0 0 0 erg=2.0
f4:n 1 2 4



## Save

```python
deck.save("model_renumbered.mcnp")
```

That's the full loop: **parse → inspect → edit → emit**, losslessly, from Python.